# VERA: conservative Qwen3.5-4B pilot on two T4 GPUs
Attach `vera-domain-v1.zip` and the locally generated `vera-kaggle-training.zip` as private Kaggle inputs. Select **GPU T4 x2** and enable Internet. Read `docs/kaggle-finetuning.md` first.
This notebook uses single-process **FP32 LoRA with model sharding**, not DDP, FP16, BF16, QLoRA, or GGUF training. Qwen3.5's FP16 stability and T4 memory fit must not be assumed. The first experiment performs five optimizer updates, including worst-case token lengths. It is not evidence of improved model quality.
The dataset remains a synthetic, unapproved candidate. No training starts until you explicitly acknowledge its limitations. The full run is disabled by default.

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from shutil import copyfile
from tempfile import mkdtemp
from zipfile import ZipFile

RUN_TIME_LIMIT_SECONDS = 1800
DEADLINE = time.monotonic() + RUN_TIME_LIMIT_SECONDS


def run_checked(command, *, env=None):
    remaining = DEADLINE - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('The approved run-time budget is exhausted')
    return subprocess.run(command, env=env, check=True, timeout=remaining)


def unique_input(*names):
    matches = [path for name in names for path in Path('/kaggle/input').rglob(name)]
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one input matching {names}; found {len(matches)}')
    return matches[0]

DATASET = unique_input('vera-domain-v1.zip', 'vera-domain-v1')
BUNDLE = unique_input('vera-kaggle-training.zip', 'vera-kaggle-training')
WORK = Path('/kaggle/working')
RUNTIME = Path(mkdtemp(prefix='vera-runtime-', dir='/tmp'))
CODE = RUNTIME / 'code'
CODE.mkdir(exist_ok=True)
modules = ('__init__', 'data', 'preflight', 'model', 'checkpoints', 'train', 'evaluation')
allowed = {'evaluate.py', 'requirements-kaggle.txt', 'runtime-contract.json', 'LICENSE'}
allowed.update(f'vera_finetuning/{name}.py' for name in modules)
if BUNDLE.is_dir():
    members = list(BUNDLE.rglob('*'))
    if any(path.is_symlink() for path in members):
        raise RuntimeError('Training code must not contain symbolic links')
    files = {path.relative_to(BUNDLE).as_posix(): path for path in members if path.is_file()}
    if set(files) != allowed or sum(path.stat().st_size for path in files.values()) > 1048576:
        raise RuntimeError('Unexpected training code contents or size')
    for name, path in files.items():
        target = CODE / name
        target.parent.mkdir(parents=True, exist_ok=True)
        copyfile(path, target)
else:
    with ZipFile(BUNDLE) as archive:
        if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
            raise RuntimeError('Unexpected files in the training bundle')
        if sum(item.file_size for item in archive.infolist()) > 1024 * 1024:
            raise RuntimeError('Training code bundle is unexpectedly large')
        archive.extractall(CODE)
CONTRACT = CODE / 'runtime-contract.json'
ENV = {**os.environ, 'PYTHONPATH': str(CODE), 'HF_HOME': str(RUNTIME / 'hf-cache'),
       'HF_HUB_CACHE': str(RUNTIME / 'hf-cache/hub'),
       'TRANSFORMERS_CACHE': str(RUNTIME / 'hf-cache/hub'),
       'HF_XET_CACHE': str(RUNTIME / 'hf-cache/xet')}
contract_summary = json.loads(CONTRACT.read_text())
print({key: contract_summary[key] for key in ('model', 'max_length', 'validated_examples')})

In [ ]:
VENV = RUNTIME / 'venv'
run_checked(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'])
run_checked([sys.executable, '-m', 'venv', '--without-pip',
             '--system-site-packages', str(VENV)])
PYTHON = str(VENV / 'bin/python')
run_checked([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '-r',
             str(CODE / 'requirements-kaggle.txt')])

In [ ]:
COMMON = ['--dataset', str(DATASET), '--contract', str(CONTRACT)]
run_checked([PYTHON, '-m', 'vera_finetuning.preflight', *COMMON,
             '--output', str(WORK / 'vera-preflight.json')], env=ENV)

## Review the preflight before continuing
Verify the split counts, token limits, assistant-only boundary and warnings. An inconsistent historical manifest prompt hash is reported separately; every actual conversation must match the captured runtime prompt. A changed runtime, including the `chat` branch's prompt, requires a rebuilt and reviewed dataset.
Set `ACKNOWLEDGE_CANDIDATE_DATA = True` only after reviewing those limitations. The pilot may stop with an out-of-memory or kernel error: retain the error and do not silently truncate examples or switch to unstable reduced precision.

In [ ]:
ACKNOWLEDGE_CANDIDATE_DATA = False
RUN_FULL_TRAINING = False
RUN_TAG = datetime.now(UTC).strftime('%Y%m%d-%H%M%S')
PILOT_OUTPUT = WORK / f'vera-pilot-{RUN_TAG}'
FULL_OUTPUT = WORK / f'vera-full-{RUN_TAG}'
FULL_RESUME = None
MAX_STEPS = None
VALIDATION_EVERY = 25
VALIDATION_EXAMPLES = 16
RUN_BEHAVIORAL_EVAL = False

if not ACKNOWLEDGE_CANDIDATE_DATA:
    raise RuntimeError('Review the candidate-data warnings and explicitly acknowledge them first')
run_checked([PYTHON, '-m', 'vera_finetuning.train', *COMMON, '--pilot',
             '--output', str(PILOT_OUTPUT), '--acknowledge-candidate-data'], env=ENV)
pilot_report = json.loads((PILOT_OUTPUT / 'report.json').read_text())
if pilot_report['steps'] != 5:
    raise RuntimeError('The pilot did not complete all five optimizer updates')
print(json.dumps(pilot_report, indent=2))

In [ ]:
if RUN_FULL_TRAINING:
    if not ACKNOWLEDGE_CANDIDATE_DATA or pilot_report['steps'] != 5:
        raise RuntimeError('Explicit consent and a successful pilot are required')
    command = [PYTHON, '-m', 'vera_finetuning.train', *COMMON, '--full',
               '--output', str(FULL_OUTPUT), '--acknowledge-candidate-data',
               '--validation-every', str(VALIDATION_EVERY),
               '--validation-examples', str(VALIDATION_EXAMPLES)]
    if FULL_RESUME is not None:
        command += ['--resume', str(FULL_RESUME)]
    if MAX_STEPS is not None:
        command += ['--max-steps', str(MAX_STEPS)]
    run_checked(command, env=ENV)
else:
    print('Full training is disabled. A successful pilot does not establish improved quality.')

In [ ]:
EVAL_PER_TASK = 3
EVAL_OUTPUT = WORK / f'vera-eval-{RUN_TAG}'
if RUN_BEHAVIORAL_EVAL:
    adapter = FULL_OUTPUT / 'adapter'
    if not adapter.is_dir():
        raise RuntimeError('Run or resume full training before behavioral evaluation')
    run_checked([PYTHON, str(CODE / 'evaluate.py'), '--dataset', str(DATASET), '--contract', str(CONTRACT),
                 '--adapter', str(adapter), '--split', 'validation',
                 '--per-task', str(EVAL_PER_TASK), '--output', str(EVAL_OUTPUT)], env=ENV)
else:
    print('Behavioral evaluation is disabled.')

## Preserve outputs and evaluate before deployment
Use Kaggle **Save Version** and retain the output directories. Each contains `report.json`, `metrics.jsonl`, an `adapter/` directory and complete checkpoints. Resume a full run from a saved `checkpoint-*` directory using a new output directory.
Training losses are only a numerical smoke check, not a quality benchmark. Run `evaluate.py` to generate base and adapter answers on a stratified validation sample, then score `answers.jsonl` locally with `finetuning/score.py`, which uses VERA's production validation. Compare behavior by task: scope, unavailable evidence, simulated evidence, citations and exact quotes. Reserve the test split for the final comparison. Never deploy based only on decreasing loss.
The adapter is not a GGUF and cannot replace the existing MTP GGUF file. Merge it with the same pinned Hugging Face base checkpoint, convert using a compatible llama.cpp version, and repeat VERA's runtime checks before selecting it.